#Transformación Bronze a Silver

In [0]:
# CONFIGURACIÓN DE PARÁMETROS

# Celda 1. Parámetros del notebook

dbutils.widgets.text("catalog_name", "banking_dev", "Catálogo")
dbutils.widgets.text("storage_account", "adlsprojectbank12", "Storage Account")
dbutils.widgets.dropdown("load_mode", "FULL", ["FULL"], "Modo de carga")

catalog_name = dbutils.widgets.get("catalog_name").strip()
storage_account = dbutils.widgets.get("storage_account").strip()
load_mode = dbutils.widgets.get("load_mode").strip().upper()

if not catalog_name or not storage_account:
    raise ValueError("catalog_name y storage_account son obligatorios.")

if load_mode != "FULL":
    raise ValueError(f"Modo no soportado: {load_mode}")

print(f"Catálogo: {catalog_name}")
print(f"Storage Account: {storage_account}")
print(f"Modo de carga: {load_mode}")

In [0]:
# CONFIGURACIÓN DE RUTAS & TABLAS

table_names = [
    "sbs_creditos_sector_economico",
    "sbs_depositos_mn",
    "sbs_depositos_me",
    "sbs_morosidad",
    "fraud_transactions",
    "customers",
    "accounts",
    "cards",
    "transactions",
    "loans",
    "payments"
]

source_tables = {name: f"{catalog_name}.bronze.{name}" for name in table_names}
target_tables = {name: f"{catalog_name}.silver.{name}" for name in table_names}

target_paths = {
    name: f"abfss://silver@{storage_account}.dfs.core.windows.net/{name}"
    for name in table_names
}

print(f"Tablas a transformar: {len(table_names)}")
for name in table_names:
    print(f"{source_tables[name]} -> {target_tables[name]}")

In [0]:
# LECTURA DE BRONZE & FUNCIONES

# Celda 3. Lectura de Bronze y funciones auxiliares

from pyspark.sql import functions as F
from pyspark.sql import types as T

bronze = {}
for name, full_name in source_tables.items():
    if not spark.catalog.tableExists(full_name):
        raise ValueError(f"No existe la tabla de origen: {full_name}")
    bronze[name] = spark.table(full_name)

#Recorta espacios y convierte cadenas vacías en NULL
def clean_text_columns(df, columns):
    for column_name in columns:
        if column_name in df.columns:
            cleaned = F.trim(F.col(column_name))
            df = df.withColumn(
                column_name,
                F.when(cleaned == "", F.lit(None)).otherwise(cleaned)
            )
    return df

#Alinea nombres y tipos con el esquema definido en Silver
def cast_to_target_schema(df, target_table):
    
    target_schema = spark.table(target_table).schema
    missing = [field.name for field in target_schema if field.name not in df.columns]
    extra = [column_name for column_name in df.columns if column_name not in [f.name for f in target_schema]]
    if missing or extra:
        raise ValueError(
            f"Desajuste de columnas en {target_table}. "
            f"Faltantes={missing}, adicionales={extra}"
        )
    return df.select(*[
        F.col(field.name).cast(field.dataType).alias(field.name)
        for field in target_schema
    ])

print("Lectura Bronze completa")
for name, df in bronze.items():
    print(f"{name}: {df.count():,} registros | {len(df.columns)} columnas")

####Transformación SBS: créditos y depósitos

In [0]:
silver = {}

# Créditos por sector económico
df = clean_text_columns(
    bronze["sbs_creditos_sector_economico"],
    ["concepto"]
)
silver["sbs_creditos_sector_economico"] = df

# Depósitos en moneda nacional
df = clean_text_columns(
    bronze["sbs_depositos_mn"],
    ["moneda", "empresa"]
)
silver["sbs_depositos_mn"] = df

# Depósitos en moneda extranjera
df = clean_text_columns(
    bronze["sbs_depositos_me"],
    ["moneda", "empresa"]
)
silver["sbs_depositos_me"] = df

print("Transformaciones SBS de créditos y depósitos completadas.")

####Transformación SBS: morosidad

In [0]:
# Los NULL de morosidad_pct representan datos no disponibles/no aplicables, no se reemplazan
df = clean_text_columns(
    bronze["sbs_morosidad"],
    ["concepto_credito", "empresa_bancaria"]
)
silver["sbs_morosidad"] = df

print("Transformación SBS morosidad completa")
print("Valores NULL de morosidad_pct se conservan")

####Transformación: fraude

In [0]:
# El dataset no dispone de una pk
# Dos transacciones pueden compartir exactamente los mismos valores
fraud_df = bronze["fraud_transactions"]

fraud_df = fraud_df.withColumn("Time", F.col("Time").cast("double"))

for column_name in [f"V{i}" for i in range(1, 29)]:
    fraud_df = fraud_df.withColumn(column_name, F.col(column_name).cast("double"))

fraud_df = (
    fraud_df
    .withColumn("Amount", F.col("Amount").cast(T.DecimalType(18, 2)))
    .withColumn("Class", F.col("Class").cast("int"))
)

silver["fraud_transactions"] = fraud_df
print("Transformación Fraud completa")

####Transformación banking: clientes, cuentas y tarjetas

In [0]:
silver["customers"] = clean_text_columns(
    bronze["customers"],
    ["document_type", "document_number", "first_name", "last_name",
     "gender", "city", "customer_segment", "customer_status"]
)

silver["accounts"] = clean_text_columns(
    bronze["accounts"],
    ["account_type", "currency", "account_status"]
)

silver["cards"] = clean_text_columns(
    bronze["cards"],
    ["card_type", "card_brand", "card_status"]
)

print("Transformaciones de customers, accounts y cards completadas.")

####Transformación banking: transacciones, préstamos y pagos

In [0]:
silver["transactions"] = clean_text_columns(
    bronze["transactions"],
    ["transaction_type", "channel", "merchant_category",
     "currency", "transaction_status"]
)

silver["loans"] = clean_text_columns(
    bronze["loans"],
    ["loan_type", "loan_status"]
)

silver["payments"] = clean_text_columns(
    bronze["payments"],
    ["payment_status"]
)

print("Transformaciones de transactions, loans y payments completadas.")

In [0]:
# VALIDACIONES DE CALIDAD

def fail_if_any(df, condition, description):
    affected = df.filter(condition).count()
    if affected > 0:
        raise ValueError(f"Validación fallida: {description}. Registros afectados: {affected:,}")
    print(f"OK: {description}")

def validate_primary_key(df, key_columns, table_name):
    fail_if_any(
        df,
        F.expr(" OR ".join([f"`{c}` IS NULL" for c in key_columns])),
        f"{table_name}: clave primaria/natural nula ({key_columns})"
    )
    duplicates = (
        df.groupBy(*key_columns)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )
    if duplicates:
        raise ValueError(
            f"{table_name}: {duplicates:,} claves duplicadas para {key_columns}"
        )
    print(f"OK: {table_name}: clave única {key_columns}")

def validate_fk(child_df, child_key, parent_df, parent_key, description, nullable=False):
    candidates = child_df.filter(F.col(child_key).isNotNull()) if nullable else child_df
    if not nullable:
        fail_if_any(child_df, F.col(child_key).isNull(), f"{description}: FK nula")
    missing = (
        candidates.select(F.col(child_key).alias("_fk")).distinct()
        .join(
            parent_df.select(F.col(parent_key).alias("_pk")).distinct(),
            F.col("_fk") == F.col("_pk"),
            "left_anti"
        ).count()
    )
    if missing:
        raise ValueError(f"{description}: {missing:,} valores FK sin correspondencia")
    print(f"OK: {description}")

def validate_nonnegative(df, columns, table_name):
    for column_name in columns:
        fail_if_any(
            df,
            F.col(column_name).isNotNull() & (F.col(column_name) < 0),
            f"{table_name}: {column_name} no puede ser negativo"
        )

def validate_required(df, columns, table_name):
    for column_name in columns:
        fail_if_any(
            df,
            F.col(column_name).isNull(),
            f"{table_name}: {column_name} es obligatorio"
        )

In [0]:
# VALIDAR SBS Y FRAUD

# SBS créditos
df = silver["sbs_creditos_sector_economico"]
validate_required(df, ["fecha_corte", "concepto", "nivel_jerarquia"], "sbs_creditos_sector_economico")
validate_primary_key(df, ["fecha_corte", "concepto", "nivel_jerarquia"], "sbs_creditos_sector_economico")
validate_nonnegative(
    df,
    ["numero_deudores", "creditos_moneda_nacional_miles_soles",
     "creditos_moneda_extranjera_miles_soles", "total_creditos_directos_miles_soles"],
    "sbs_creditos_sector_economico"
)
fail_if_any(
    df,
    F.col("distribucion_creditos_directos_pct").isNotNull() &
    ((F.col("distribucion_creditos_directos_pct") < 0) |
     (F.col("distribucion_creditos_directos_pct") > 100)),
    "sbs_creditos_sector_economico: distribución fuera de 0–100"
)

# SBS depósitos
for table_name in ["sbs_depositos_mn", "sbs_depositos_me"]:
    df = silver[table_name]
    validate_required(df, ["fecha_corte", "moneda", "empresa"], table_name)
    validate_primary_key(df, ["fecha_corte", "empresa", "moneda"], table_name)
    validate_nonnegative(
        df,
        ["numero_cuentas", "saldo_anterior_miles_soles", "abonos_miles_soles",
         "intereses_capitalizados_miles_soles", "retiros_cargos_miles_soles",
         "saldo_final_miles_soles"],
        table_name
    )

# SBS morosidad: porcentaje NULL permitido
df = silver["sbs_morosidad"]
validate_required(df, ["fecha_corte", "concepto_credito", "empresa_bancaria"], "sbs_morosidad")
validate_primary_key(
    df, ["fecha_corte", "concepto_credito", "empresa_bancaria"], "sbs_morosidad"
)
fail_if_any(
    df,
    F.col("morosidad_pct").isNotNull() &
    ((F.col("morosidad_pct") < 0) | (F.col("morosidad_pct") > 100)),
    "sbs_morosidad: morosidad fuera de 0–100"
)

# Fraude
df = silver["fraud_transactions"]
fail_if_any(df, F.col("Class").isNull(), "fraud_transactions: Class nulo")
fail_if_any(
    df, ~F.col("Class").isin(0, 1),
    "fraud_transactions: Class debe ser 0 o 1"
)
validate_nonnegative(df, ["Amount"], "fraud_transactions")

print("Validaciones SBS y Fraud completadas.")

In [0]:
# VALIDAR BANKING

# Claves primarias
for table_name, key in [
    ("customers", ["customer_id"]),
    ("accounts", ["account_id"]),
    ("cards", ["card_id"]),
    ("transactions", ["transaction_id"]),
    ("loans", ["loan_id"]),
    ("payments", ["payment_id"])
]:
    validate_primary_key(silver[table_name], key, table_name)

# Campos obligatorios y relaciones
validate_required(
    silver["customers"],
    ["document_type", "document_number", "first_name", "last_name",
     "birth_date", "registration_date"],
    "customers"
)
validate_required(
    silver["accounts"],
    ["customer_id", "account_type", "currency", "opening_date"],
    "accounts"
)
validate_required(
    silver["cards"],
    ["account_id", "card_type", "card_brand", "issue_date", "expiration_date"],
    "cards"
)
validate_required(
    silver["transactions"],
    ["account_id", "transaction_date", "transaction_type", "amount",
     "currency", "transaction_status", "is_fraud"],
    "transactions"
)
validate_required(
    silver["loans"],
    ["customer_id", "loan_type", "principal_amount", "interest_rate",
     "term_months", "disbursement_date", "due_date"],
    "loans"
)
validate_required(
    silver["payments"],
    ["loan_id", "payment_date", "payment_amount", "principal_paid",
     "interest_paid", "payment_status"],
    "payments"
)

validate_fk(silver["accounts"], "customer_id", silver["customers"], "customer_id",
            "accounts.customer_id -> customers.customer_id")
validate_fk(silver["cards"], "account_id", silver["accounts"], "account_id",
            "cards.account_id -> accounts.account_id")
validate_fk(silver["transactions"], "account_id", silver["accounts"], "account_id",
            "transactions.account_id -> accounts.account_id")
validate_fk(silver["transactions"], "card_id", silver["cards"], "card_id",
            "transactions.card_id -> cards.card_id", nullable=True)
validate_fk(silver["loans"], "customer_id", silver["customers"], "customer_id",
            "loans.customer_id -> customers.customer_id")
validate_fk(silver["payments"], "loan_id", silver["loans"], "loan_id",
            "payments.loan_id -> loans.loan_id")

# Una tarjeta informada en una transacción debe pertenecer a la misma cuenta
tx_with_card = silver["transactions"].filter(F.col("card_id").isNotNull()).alias("t")
cards_ref = silver["cards"].select("card_id", "account_id").alias("c")
invalid_card_account = tx_with_card.join(
    cards_ref,
    F.col("t.card_id") == F.col("c.card_id"),
    "inner"
).filter(F.col("t.account_id") != F.col("c.account_id")).count()
if invalid_card_account:
    raise ValueError(
        f"transactions: {invalid_card_account:,} tarjetas no corresponden a la cuenta de la transacción"
    )
print("OK: correspondencia entre tarjeta y cuenta en transactions")

# Reglas de dominio y coherencia temporal
validate_nonnegative(silver["loans"], ["principal_amount", "interest_rate"], "loans")
validate_nonnegative(
    silver["payments"], ["payment_amount", "principal_paid", "interest_paid"], "payments"
)
validate_nonnegative(silver["transactions"], ["amount"], "transactions")

fail_if_any(
    silver["loans"],
    (F.col("term_months") <= 0) | (F.col("due_date") < F.col("disbursement_date")),
    "loans: plazo positivo y vencimiento no anterior al desembolso"
)
fail_if_any(
    silver["cards"],
    F.col("expiration_date") < F.col("issue_date"),
    "cards: vencimiento no anterior a emisión"
)
fail_if_any(
    silver["customers"],
    F.col("birth_date") > F.col("registration_date"),
    "customers: nacimiento posterior al registro"
)
fail_if_any(
    silver["transactions"],
    ~F.col("is_fraud").isin(0, 1),
    "transactions: is_fraud debe ser 0 o 1"
)

print("Validaciones de las seis tablas bancarias completadas.")

In [0]:
# VALIDAR TABLAS DELTA DESTINO Y ESQUEMAS

silver_ready = {}

for table_name in table_names:
    target_table = target_tables[table_name]

    if not spark.catalog.tableExists(target_table):
        raise ValueError(f"No existe la tabla destino: {target_table}")

    detail = spark.sql(f"DESCRIBE DETAIL {target_table}").first()
    if detail["format"].lower() != "delta":
        raise ValueError(f"La tabla destino no es Delta: {target_table}")

    expected_location = target_paths[table_name].rstrip("/")
    actual_location = detail["location"].rstrip("/")
    if actual_location.lower() != expected_location.lower():
        raise ValueError(
            f"LOCATION inesperada en {target_table}. "
            f"Esperada={expected_location}; actual={actual_location}"
        )

    silver_ready[table_name] = cast_to_target_schema(
        silver[table_name], target_table
    )

    print(f"Destino validado: {target_table} | {actual_location}")

print("Los 11 destinos y esquemas fueron validados. Aún no se ha sobrescrito Silver.")


In [0]:
# CARGA FULL A SILVER

# Se ejecuta únicamente después de completar todas las validaciones anteriores
# Cada tabla se sobrescribe en su ubicación Delta externa ya registrada en Unity Catalog
for table_name in table_names:
    target_table = target_tables[table_name]
    (
        silver_ready[table_name]
        .write
        .format("delta")
        .mode("overwrite")
        .insertInto(target_table)
    )
    print(f"Carga completada: {target_table}")

print("Carga FULL Bronze a Silver finalizada")

In [0]:
# VALIDACIÓN DE CARGA

summary = []

for table_name in table_names:
    source_count = bronze[table_name].count()
    target_count = spark.table(target_tables[table_name]).count()

    status = "SUCCESS" if source_count == target_count else "FAILED"
    summary.append((table_name, source_count, target_count, status))

    if status != "SUCCESS":
        raise ValueError(
            f"Conteos distintos en {table_name}: Bronze={source_count:,}, Silver={target_count:,}"
        )

summary_df = spark.createDataFrame(
    summary,
    ["table_name", "bronze_count", "silver_count", "status"]
)

display(summary_df.orderBy("table_name"))

print("Todas las tablas Silver tienen el mismo número de registros que Bronze.")